In [6]:
# If needed, run once in a notebook:
# %pip install -q pandas numpy requests yfinance lxml

from datetime import date
import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

AS_OF = pd.Timestamp("2026-08-21")


def close_series(ticker, start, end):
    """Download one index/security and return a clean daily Close Series."""
    data = yf.download(
        ticker,
        start=pd.Timestamp(start).strftime("%Y-%m-%d"),
        # yfinance's end date is exclusive, so add one day for an inclusive end.
        end=(pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=False,
    )
    if data.empty:
        raise ValueError(f"No data returned for {ticker}")
    close = data["Close"]
    if isinstance(close, pd.DataFrame):  # compatible with recent yfinance versions
        close = close.iloc[:, 0]
    close = close.dropna().sort_index()
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close


def period_return(ticker, start, end):
    close = close_series(ticker, start, end)
    return close.iloc[-1] / close.iloc[0] - 1


## Question 1 — S&P 500 stocks added to the index


In [7]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (compatible; SMA-Zoomcamp/2026)"}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(response.text)
sp500 = tables[0].copy()
sp500.columns = [str(column).strip() for column in sp500.columns]
sp500 = sp500.rename(
    columns={
        "Symbol": "ticker",
        "Security": "company",
        "Date added": "date_added",
    }
)
sp500["date_added"] = pd.to_datetime(sp500["date_added"], errors="coerce")
sp500["addition_year"] = sp500["date_added"].dt.year

additions_since_2020 = (
    sp500.loc[sp500["addition_year"].ge(2020), "addition_year"]
    .value_counts()
    .sort_index()
    .rename_axis("year")
    .rename("additions")
    .to_frame()
)
print(additions_since_2020)
print(
    "Q1 answer — year with most additions since 2020:",
    additions_since_2020["additions"].idxmax(),
)

# "More than 20 years" means added before the date 20 years before today.
twenty_year_cutoff = pd.Timestamp.today().normalize() - pd.DateOffset(years=20)
print(
    "Additional — current constituents added more than 20 years ago:",
    int(sp500["date_added"].lt(twenty_year_cutoff).sum()),
)


      additions
year           
2020         10
2021         10
2022         15
2023         15
2024         16
2025         18
2026         13
Q1 answer — year with most additions since 2020: 2025
Additional — current constituents added more than 20 years ago: 224


/tmp/ipykernel_5364/2632526135.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


## Question 2 — YTD returns for world indexes

The assignment lists 11 indexes (the question says "out of 10", but the list
contains 11). This uses the first available close on/after January 1 and the
last available close on/before August 21 for every ticker.


In [8]:
tickers = {
    "United States S&P 500": "^GSPC",
    "China Shanghai Composite": "000001.SS",
    "Hong Kong Hang Seng": "^HSI",
    "Australia ASX 200": "^AXJO",
    "India Nifty 50": "^NSEI",
    "Canada TSX Composite": "^GSPTSE",
    "Germany DAX": "^GDAXI",
    "United Kingdom FTSE 100": "^FTSE",
    "Japan Nikkei 225": "^N225",
    "Mexico IPC Mexico": "^MXX",
    "Brazil Ibovespa": "^BVSP",
}

ytd_rows = []
for name, ticker in tickers.items():
    close = close_series(ticker, "2026-01-01", AS_OF)
    ytd_rows.append(
        {
            "index": name,
            "ticker": ticker,
            "first_date": close.index[0],
            "last_date": close.index[-1],
            "start_close": close.iloc[0],
            "end_close": close.iloc[-1],
            "ytd_return": close.iloc[-1] / close.iloc[0] - 1,
        }
    )

ytd = pd.DataFrame(ytd_rows).sort_values("ytd_return", ascending=False)
us_return = ytd.loc[ytd["ticker"].eq("^GSPC"), "ytd_return"].iloc[0]
ytd["better_than_us"] = ytd["ytd_return"] > us_return
print(ytd)
print("Q2 answer — US S&P 500 YTD return:", us_return)
print("Q2 answer — indexes better than S&P 500:", int(ytd["better_than_us"].sum()))

# Additional: compare performance over 3, 5, and 10 years ending on AS_OF.
periods = {"3_year": AS_OF - pd.DateOffset(years=3),
           "5_year": AS_OF - pd.DateOffset(years=5),
           "10_year": AS_OF - pd.DateOffset(years=10)}
long_term = []
for period_name, start in periods.items():
    for name, ticker in tickers.items():
        long_term.append(
            {
                "period": period_name,
                "index": name,
                "ticker": ticker,
                "return": period_return(ticker, start, AS_OF),
            }
        )
long_term = pd.DataFrame(long_term)
us_long_term = long_term[long_term["ticker"].eq("^GSPC")].set_index("period")["return"]
long_term["better_than_us"] = long_term.apply(
    lambda row: row["return"] > us_long_term.loc[row["period"]], axis=1
)
print(long_term.pivot(index="index", columns="period", values="return"))
print(long_term.groupby("period")["better_than_us"].sum())


                       index     ticker first_date  last_date  start_close  \
8           Japan Nikkei 225      ^N225 2026-01-05 2026-08-21  51,832.8008   
5       Canada TSX Composite    ^GSPTSE 2026-01-02 2026-08-21  31,883.4004   
0      United States S&P 500      ^GSPC 2026-01-02 2026-08-21   6,858.4702   
7    United Kingdom FTSE 100      ^FTSE 2026-01-02 2026-08-21   9,951.0996   
10           Brazil Ibovespa      ^BVSP 2026-01-02 2026-08-21 160,539.0000   
6                Germany DAX     ^GDAXI 2026-01-02 2026-08-21  24,539.3398   
3          Australia ASX 200      ^AXJO 2026-01-02 2026-08-21   8,727.7998   
9          Mexico IPC Mexico       ^MXX 2026-01-02 2026-08-21  64,141.3594   
2        Hong Kong Hang Seng       ^HSI 2026-01-02 2026-08-21  26,338.4707   
1   China Shanghai Composite  000001.SS 2026-01-05 2026-08-21   4,023.4170   
4             India Nifty 50      ^NSEI 2026-01-01 2026-08-21  26,146.5508   

      end_close  ytd_return  better_than_us  
8   66,016.3594  

## Question 3 — S&P 500 corrections

A correction is represented as one drawdown episode: it starts after a
previous all-time high and ends when the index reaches a new high. Duration
below is calendar days from the peak to the trough.


In [9]:
spx = close_series("^GSPC", "1950-01-01", AS_OF).rename("close").to_frame()
spx["running_high"] = spx["close"].cummax()
spx["drawdown"] = spx["close"] / spx["running_high"] - 1
spx["in_drawdown"] = spx["drawdown"] < 0

# A new group begins whenever the market switches between recovery and decline.
spx["episode"] = spx["in_drawdown"].ne(spx["in_drawdown"].shift()).cumsum()
corrections = []
for _, episode in spx[spx["in_drawdown"]].groupby("episode"):
    trough_date = episode["close"].idxmin()
    before_episode = spx.loc[:episode.index[0]].iloc[:-1]
    peak_close = before_episode["running_high"].iloc[-1]
    peak_date = before_episode.index[before_episode["close"].eq(peak_close)][-1]
    trough_close = episode.loc[trough_date, "close"]
    episode_end_position = spx.index.get_loc(episode.index[-1])
    recovery_date = (
        spx.index[episode_end_position + 1]
        if episode_end_position + 1 < len(spx)
        else pd.NaT
    )
    corrections.append(
        {
            "peak_date": peak_date,
            "trough_date": trough_date,
            "recovery_date": recovery_date,
            "peak_close": peak_close,
            "trough_close": trough_close,
            "drawdown": trough_close / peak_close - 1,
            "duration_calendar_days": (trough_date - peak_date).days,
        }
    )

corrections = pd.DataFrame(corrections)
corrections = corrections[corrections["drawdown"] <= -0.05].copy()
corrections = corrections.sort_values("drawdown")
print(corrections.head(10))
print(
    "Q3 answer — drawdown percentiles:",
    corrections["drawdown"].abs().quantile([0.25, 0.50, 0.75]).to_dict(),
)
print(
    "Q3 answer — duration percentiles:",
    corrections["duration_calendar_days"].quantile([0.25, 0.50, 0.75]).to_dict(),
)


     peak_date trough_date recovery_date  peak_close  trough_close  drawdown  \
448 2007-10-09  2009-03-09    2013-03-28  1,565.1500      676.5300   -0.5678   
443 2000-03-24  2002-10-09    2007-05-30  1,527.4600      776.7600   -0.4915   
206 1973-01-11  1974-10-03    1980-07-17    120.2400       62.2800   -0.4820   
193 1968-11-29  1970-05-26    1972-03-06    108.3700       69.2900   -0.3606   
574 2020-02-19  2020-03-23    2020-08-18  3,386.1499    2,237.3999   -0.3392   
292 1987-08-25  1987-12-04    1989-07-26    336.7700      223.9200   -0.3351   
133 1961-12-12  1962-06-26    1963-09-03     72.6400       52.3200   -0.2797   
219 1980-11-28  1982-08-12    1982-11-03    140.5200      102.4200   -0.2711   
620 2022-01-03  2022-10-12    2024-01-19  4,796.5601    3,577.0300   -0.2543   
176 1966-02-09  1966-10-07    1967-05-04     94.0600       73.2000   -0.2218   

     duration_calendar_days  
448                     517  
443                     929  
206                     630  

## Question 4 — Amazon earnings surprises


In [12]:
amazon = yf.Ticker("AMZN")
earnings = amazon.get_earnings_dates().reset_index()

print("Available columns:")
print(earnings.columns.tolist())

# Rename the earnings-date column
earnings = earnings.rename(
    columns={earnings.columns[0]: "earnings_date"}
)

# Convert dates safely, regardless of timezone format
earnings["earnings_date"] = (
    pd.to_datetime(
        earnings["earnings_date"],
        utc=True,
        errors="coerce",
    )
    .dt.tz_localize(None)
)

# Find the surprise column.
# Depending on the yfinance version, it may be:
# "Surprise %", "Surprise(%)", or another similar name.
surprise_columns = [
    column
    for column in earnings.columns
    if "surprise" in str(column).lower()
]

if not surprise_columns:
    raise KeyError(
        "No earnings surprise column found. "
        f"Available columns: {earnings.columns.tolist()}"
    )

original_surprise_column = surprise_columns[0]

# Use one consistent internal name
earnings = earnings.rename(
    columns={
        original_surprise_column: "surprise_pct"
    }
)

# Convert the surprise values to numbers.
# This removes the future earnings entry if its value is empty.
earnings["surprise_pct"] = pd.to_numeric(
    earnings["surprise_pct"],
    errors="coerce",
)

earnings = earnings.dropna(
    subset=["earnings_date", "surprise_pct"]
).copy()


# Download Amazon closing prices
amzn_close = (
    close_series(
        "AMZN",
        "2019-01-01",
        pd.Timestamp.today(),
    )
    .rename("close")
)

# Make sure the price index is timezone-naive
if amzn_close.index.tz is not None:
    amzn_close.index = amzn_close.index.tz_localize(None)

amzn_close.index = pd.to_datetime(
    amzn_close.index
).normalize()

amzn_close = amzn_close.sort_index()


# Match earnings dates with trading dates
events = []

for _, row in earnings.iterrows():
    announcement_date = row["earnings_date"].normalize()

    # Use the first trading day on or after the earnings date
    announcement_position = amzn_close.index.searchsorted(
        announcement_date,
        side="left",
    )

    # We need one trading day before and one after
    if (
        announcement_position < 1
        or announcement_position + 1 >= len(amzn_close)
    ):
        continue

    day1 = amzn_close.index[announcement_position - 1]
    day2 = amzn_close.index[announcement_position]
    day3 = amzn_close.index[announcement_position + 1]

    two_day_return = (
        amzn_close.iloc[announcement_position + 1]
        / amzn_close.iloc[announcement_position - 1]
        - 1
    )

    events.append(
        {
            "earnings_date": row["earnings_date"],
            "trading_day": day2,
            "day_before": day1,
            "day_after": day3,
            "surprise_pct": row["surprise_pct"],
            "two_day_return": two_day_return,
        }
    )


# Create final event DataFrame
events = pd.DataFrame(events)

if events.empty:
    raise ValueError(
        "No earnings dates could be matched to Amazon prices."
    )

events = (
    events
    .drop_duplicates(subset=["earnings_date"])
    .sort_values("earnings_date", ascending=False)
)

positive = events[
    events["surprise_pct"] > 0
].copy()


# Display results
print("Matched earnings events:")
display(events)

print("Positive-surprise events:")
display(positive)

print(
    "Q4 answer — positive-surprise median two-day return:",
    f"{positive['two_day_return'].median():.4%}",
)

correlation = positive[
    ["surprise_pct", "two_day_return"]
].corr().loc[
    "surprise_pct",
    "two_day_return",
]

print(
    "Q4 answer — surprise/return correlation:",
    f"{correlation:.4f}",
)

Available columns:
['Earnings Date', 'EPS Estimate', 'Reported EPS', 'Surprise(%)']
Matched earnings events:


,earnings_date,trading_day,day_before,day_after,surprise_pct,two_day_return
0,2026-07-30 20:00:00,2026-07-30,2026-07-29,2026-07-31,215.0200,0.1982
1,2026-04-29 20:00:00,2026-04-29,2026-04-28,2026-04-30,69.0200,0.0206
2,2026-02-05 21:00:00,2026-02-05,2026-02-04,2026-02-06,0.2200,-0.0973
3,2025-10-30 20:00:00,2025-10-30,2025-10-29,2025-10-31,25.2000,0.0604
4,2025-07-31 20:00:00,2025-07-31,2025-07-30,2025-08-01,27.1900,-0.0671
5,2025-05-01 20:00:00,2025-05-01,2025-04-30,2025-05-02,16.7700,0.0301
6,2025-02-06 21:00:00,2025-02-06,2025-02-05,2025-02-07,25.2900,-0.0297
7,2024-10-31 20:00:00,2024-10-31,2024-10-30,2024-11-01,25.2200,0.0270
8,2024-08-01 16:00:00,2024-08-01,2024-07-31,2024-08-02,23.7600,-0.1020
9,2024-04-30 20:00:00,2024-04-30,2024-04-29,2024-05-01,17.6700,-0.0108


Positive-surprise events:


,earnings_date,trading_day,day_before,day_after,surprise_pct,two_day_return
0,2026-07-30 20:00:00,2026-07-30,2026-07-29,2026-07-31,215.0200,0.1982
1,2026-04-29 20:00:00,2026-04-29,2026-04-28,2026-04-30,69.0200,0.0206
2,2026-02-05 21:00:00,2026-02-05,2026-02-04,2026-02-06,0.2200,-0.0973
3,2025-10-30 20:00:00,2025-10-30,2025-10-29,2025-10-31,25.2000,0.0604
4,2025-07-31 20:00:00,2025-07-31,2025-07-30,2025-08-01,27.1900,-0.0671
5,2025-05-01 20:00:00,2025-05-01,2025-04-30,2025-05-02,16.7700,0.0301
6,2025-02-06 21:00:00,2025-02-06,2025-02-05,2025-02-07,25.2900,-0.0297
7,2024-10-31 20:00:00,2024-10-31,2024-10-30,2024-11-01,25.2200,0.0270
8,2024-08-01 16:00:00,2024-08-01,2024-07-31,2024-08-02,23.7600,-0.1020
9,2024-04-30 20:00:00,2024-04-30,2024-04-29,2024-05-01,17.6700,-0.0108


Q4 answer — positive-surprise median two-day return: 0.3528%
Q4 answer — surprise/return correlation: 0.3306


## Optional Question 5 — capstone idea

Replace this example with your own project idea in the homework form.


In [15]:
capstone_idea = """
I want to build a simple model that predicts whether to buy, hold, 
or avoid a selected company’s stock over the next month using its recent price trend,
trading volume, volatility, and earnings results.
"""
print(capstone_idea)



I want to build a simple model that predicts whether to buy, hold, 
or avoid a selected company’s stock over the next month using its recent price trend,
trading volume, volatility, and earnings results.



In [16]:
## Optional Question 6 — additional metrics

print("""
For my project, I would investigate the following metrics for Amazon:

- Price momentum: 20-day and 50-day returns from Yahoo Finance to identify the stock's trend.
- Volatility: 20-day rolling standard deviation of daily returns to measure risk.
- Trading volume: Current volume compared with the 20-day average to measure investor interest.
- RSI: To identify potentially overbought or oversold conditions.
- Earnings surprise: Reported EPS compared with expected EPS using yfinance.
- Market volatility: VIX (VIXCLS) from FRED to measure overall market stress.
""")


For my project, I would investigate the following metrics for Amazon:

- Price momentum: 20-day and 50-day returns from Yahoo Finance to identify the stock's trend.
- Volatility: 20-day rolling standard deviation of daily returns to measure risk.
- Trading volume: Current volume compared with the 20-day average to measure investor interest.
- RSI: To identify potentially overbought or oversold conditions.
- Earnings surprise: Reported EPS compared with expected EPS using yfinance.
- Market volatility: VIX (VIXCLS) from FRED to measure overall market stress.

